# Lending Club EDA — Data Understanding & Structural Profiling

**Purpose**
- Confirm table row counts at every pipeline stage
- Split all 151 raw columns into numeric vs. categorical (by cast-success rate)
- Profile null% and cardinality for every column

**Where this fits**

| | |
|---|---|
| Position | 1 of 14 EDA notebooks under `notebooks/02_eda/` |
| Data access | Read-only against `data/02_interim/lendingclub.duckdb` |
| Cleaning | None here — happens later, in `notebooks/03_data_cleaning/` |
| Convention | Every code cell has markdown before it (what/why/how) and after it (what the output means, what's next) |

## Cell map

| # | Step |
|---|---|
| 1 | Connect, confirm row counts (raw / matured / windowed) |
| 2 | Numeric vs. categorical split, all 151 raw columns |
| 3 | Eyeball full column lists — sample values, cardinality |
| 4 | Tag date and time-period columns |
| 5 | Flag ambiguous columns for the cleaning stage, by row number |
| 6 | Full null% / cardinality profile (`SUMMARIZE`) |
| 7 | Check whether high-missingness columns carry signal |

**Scope**
- Profiles all 151 raw columns — none picked or dropped yet
- Starting with `02_eda/02_data_quality_integrity.ipynb`, every later notebook narrows to a fixed, named shortlist (see that notebook's cell 1) — that shortlist, not this notebook, drives the rest of the EDA suite

## Cell 1 — Connect & confirm row counts

- Open the interim DuckDB file read-only
- Confirm `raw_mat`, `matured`, `windowed` row counts match what the ingestion notebook produced — a sanity check before profiling anything

**Answers:** do the three staged tables have the row counts the ingestion notebook produced?

In [1]:
import sys, os, duckdb, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath("../_shared"))
from nb_setup import connect

con, ASSETS_TABLES, ASSETS_PLOTS = connect()  # read-only; creates the asset folders if missing

# row counts at each pipeline stage, plus raw column count
for tbl in ["raw_mat", "matured", "windowed"]:
    n = con.sql(f"SELECT count(*) FROM {tbl}").fetchone()[0]
    print(f"{tbl}: {n:,} rows")
n_cols = len(con.sql("DESCRIBE raw_mat").fetchall())
print(f"raw_mat: {n_cols} columns")

raw_mat: 2,260,701 rows
matured: 1,348,099 rows
windowed: 1,195,879 rows
raw_mat: 151 columns


**Result**

| Table | Rows |
|---|---|
| `raw_mat` | 2,260,701 |
| `matured` | 1,348,099 |
| `windowed` | 1,195,879 |

- 151 raw columns confirmed on `raw_mat` — matches the ingestion notebook's output
- Every downstream notebook works from a subset of these

**Next:** split columns into numeric vs. categorical — nothing arrived typed (everything loaded as `VARCHAR` on purpose, so nothing gets silently mis-cast).

## Cell 2 — Numeric vs. categorical, by cast-success rate

- For every column in `windowed`, check what fraction of non-null values `TRY_CAST`s to `DOUBLE`
- More reliable than trusting column names; flags genuinely ambiguous columns

**Answers:** how many columns are numeric, how many categorical, and are any ambiguous?

In [2]:
cols = [r[0] for r in con.sql("DESCRIBE windowed").fetchall()]
rows = []
# per column: null%, distinct count, and cast-to-DOUBLE success rate
for c in cols:
    r = con.sql(f'''
        SELECT count(*) n, count("{c}") n_notnull,
               count(DISTINCT "{c}") n_distinct,
               count(TRY_CAST("{c}" AS DOUBLE)) n_castable
        FROM windowed
    ''').fetchone()
    n, n_notnull, n_distinct, n_castable = r
    cast_rate = n_castable / n_notnull if n_notnull else 0
    rows.append((c, n_notnull / n, n_distinct, cast_rate))
type_df = pd.DataFrame(rows, columns=["column", "pct_notnull", "n_distinct", "cast_rate"])
# >95% castable -> numeric, <5% -> categorical/text, else flagged mixed (worth a manual look)
type_df["inferred_type"] = np.where(type_df["cast_rate"] > 0.95, "numeric",
                             np.where(type_df["cast_rate"] < 0.05, "categorical/text", "mixed"))
print(type_df["inferred_type"].value_counts())
print()
print("mixed columns (worth a manual look):")
print(type_df[type_df["inferred_type"] == "mixed"].to_string(index=False))

inferred_type
numeric             114
categorical/text     38
Name: count, dtype: int64

mixed columns (worth a manual look):
Empty DataFrame
Columns: [column, pct_notnull, n_distinct, cast_rate, inferred_type]
Index: []


**Result**

| Type | Count |
|---|---|
| Numeric | 114 |
| Categorical/text | 38 |
| Mixed (ambiguous) | 0 |

No columns landed in the "mixed" bucket — confirms the type assumptions the cleaning pipeline and every other EDA notebook rely on.

**Next:** the cast-rate threshold is mechanical — it can still misclassify a column a human would recognize immediately (e.g. a numeric-looking ID or code). Eyeballing the actual lists, then flagging anything that looks wrong, catches what the automated split can't.

## Cell 3 — Eyeball the full column lists

- The 95%/5% cast-rate threshold is mechanical — it can't tell a genuine numeric measure from a numeric-looking identifier or code, and cardinality alone doesn't either (a rare-event count can have as few distinct values as a real code)
- Printing every column with a few actual sample values and its distinct-value count lets a human check both at once, before moving on

**Answers:** looking at real sample values and cardinality, does anything in the numeric or categorical bucket look miscategorized?

In [3]:
# full column lists, one bucket at a time
numeric_cols = sorted(type_df.loc[type_df["inferred_type"] == "numeric", "column"])
categorical_cols = sorted(type_df.loc[type_df["inferred_type"] == "categorical/text", "column"])

def sample_values(col, k=4, width=35):
    # unique (DISTINCT), unseeded -- different sample each run; padded with "" if the
    # column has fewer than k distinct non-null values (itself a useful signal)
    vals = con.sql(f'''
        SELECT val FROM (SELECT DISTINCT "{col}" AS val FROM windowed WHERE "{col}" IS NOT NULL)
        USING SAMPLE {k} ROWS
    ''').df()["val"].tolist()
    vals = vals + [""] * (k - len(vals))
    def clip(v):
        s = str(v)
        return s if len(s) <= width else s[:width - 3] + "..."
    return [clip(v) for v in vals]

def build_preview(col_list):
    records = []
    for c in col_list:
        n_distinct = con.sql(f'SELECT count(DISTINCT "{c}") FROM windowed').fetchone()[0]
        records.append((c, *sample_values(c), n_distinct, "No"))
    return pd.DataFrame(records, columns=["column", "sample_1", "sample_2", "sample_3", "sample_4", "n_distinct", "is_ambiguous"])

numeric_preview_df = build_preview(numeric_cols)
categorical_preview_df = build_preview(categorical_cols)

print(f"numeric columns ({len(numeric_preview_df)}):")
print(numeric_preview_df.to_string())
print()
print(f"categorical columns ({len(categorical_preview_df)}):")
print(categorical_preview_df.to_string())

numeric columns (114):
                                         column  sample_1       sample_2         sample_3          sample_4  n_distinct is_ambiguous
0                                acc_now_delinq       1.0            3.0              0.0               6.0           8           No
1                          acc_open_past_24mths      29.0           32.0             23.0              10.0          55           No
2                                      all_util     140.0           34.0            106.0               0.0         169           No
3                                    annual_inc  228490.0        53654.0          28000.0          132533.0       59182           No
4                              annual_inc_joint  105240.0       108900.0         425000.0           43500.0        4052           No
5                                   avg_cur_bal   10223.0        30840.0           8643.0           13346.0       75220           No
6                                bc_open_to_bu

**Result**

Two columns in the numeric bucket look miscategorized once actual values and cardinality are visible:

| Column | n_distinct | Why it's not really numeric |
|---|---|---|
| `id` | ~1.2M (nearly all unique) | A loan identifier — casts to `DOUBLE` because it's stored as digits, but it's a label, not a magnitude |
| `policy_code` | 1 | A single-valued category code in this population, not a measured quantity |

One more worth naming explicitly: `is_bad` also sits in the numeric bucket (it casts cleanly, 0/1) — but it's the modeling target defined in the ingestion notebook, not a candidate feature. It's excluded from feature consideration by definition, not because anything about it needs flagging.

Everything else, including the other low-`n_distinct` numeric columns (e.g. `num_tl_30dpd`, `inq_last_6mths`, `acc_now_delinq`), reads as expected: genuine counts of a rare event, not category codes — small range because the event is rare, not because the field is secretly categorical.

**Next:** a few categorical columns are also worth a closer look — some are dates or durations in disguise.

## Cell 4 — Tag date and time-period columns

- Some categorical columns aren't general text — they're dates or durations, just not parsed as such (parsing/casting is a cleaning-stage decision, not an EDA one)
- Printing actual raw values before tagging, so the tag is evidenced, not assumed
- This is additive: `inferred_type` (numeric/categorical, from the cast-rate test) stays exactly as computed — `domain_type` is a separate note layered on top

**Answers:** which categorical columns are actually dates, which are durations, and what does their raw format actually look like?

In [4]:
DATE_LIKE = ["earliest_cr_line", "issue_d", "last_credit_pull_d", "last_pymnt_d", "sec_app_earliest_cr_line"]
TIME_PERIOD_LIKE = ["emp_length", "term"]

for c in DATE_LIKE + TIME_PERIOD_LIKE:
    sample = con.sql(f'SELECT "{c}" FROM windowed WHERE "{c}" IS NOT NULL LIMIT 6').df()[c].tolist()
    print(f"{c}: {sample}")

categorical_preview_df["domain_type"] = "categorical"
categorical_preview_df.loc[categorical_preview_df["column"].isin(DATE_LIKE), "domain_type"] = "date"
categorical_preview_df.loc[categorical_preview_df["column"].isin(TIME_PERIOD_LIKE), "domain_type"] = "time_period"
print()
print(categorical_preview_df.to_string())

earliest_cr_line: ['May-2002', 'Dec-2010', 'Jan-2006', 'Jul-1994', 'Aug-1997', 'Jun-1989']
issue_d: ['May-2016', 'May-2016', 'May-2016', 'May-2016', 'May-2016', 'May-2016']
last_credit_pull_d: ['Mar-2019', 'Feb-2017', 'Jan-2019', 'Nov-2016', 'Mar-2019', 'Feb-2019']
last_pymnt_d: ['Feb-2019', 'Feb-2017', 'Jan-2019', 'Oct-2016', 'Jan-2017', 'Sep-2018']
sec_app_earliest_cr_line: ['Aug-1990', 'Jul-2008', 'Jul-2007', 'Apr-2014', 'Apr-2004', 'Mar-2004']
emp_length: ['4 years', '3 years', '9 years', '10+ years', '2 years', '10+ years']
term: [' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months', ' 36 months']

                       column                             sample_1                             sample_2                             sample_3                             sample_4  n_distinct is_ambiguous  domain_type
0                  addr_state                                   CO                                   KY                                   ND                 

**Result**

| Group | Columns | Raw format |
|---|---|---|
| `date` | `earliest_cr_line`, `issue_d`, `last_credit_pull_d`, `last_pymnt_d`, `sec_app_earliest_cr_line` | `Mon-YYYY` (month abbreviation + year, no day) |
| `time_period` | `emp_length`, `term` | Free-text duration (`"10+ years"`, `"36 months"`) |

All five date-like columns share the same raw text format — consistent with being genuine dates that just haven't been parsed yet. None of this changes `inferred_type`; both groups still correctly read as `categorical/text` by the cast-rate test, since none of these strings cast to `DOUBLE`.

**Next:** record `id` and `policy_code` — the two genuinely miscategorized columns — somewhere the cleaning notebook can actually check.

## Cell 5 — Flag ambiguous columns for the cleaning stage, by row number

- A markdown note here is easy to miss; a file the cleaning notebook actually reads is not
- Referencing rows from the tables printed in cell 3 (by table + row number) is less error-prone than retyping column names by hand
- Writes the flagged column + reason to `eda01_ambiguous_overrides.csv`, so `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` can act on it instead of relying on the cast-rate split alone

**Answers:** which columns need to be handled by note, not by cast-rate, and why?

In [5]:
# flag ambiguous columns by row number from the tables printed in cell 3:
# (source table, row number, comment)
FLAGGED_ROWS = [
    ("numeric", 25, "loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test"),
    ("numeric", 77, "single-valued category code in this population, not a measured quantity -- treat as categorical"),
]

source_map = {"numeric": numeric_preview_df, "categorical": categorical_preview_df}
overrides = []
for source, row, note in FLAGGED_ROWS:
    col_name = source_map[source].loc[row, "column"]
    source_map[source].loc[row, "is_ambiguous"] = "Yes"
    overrides.append((col_name, note))

overrides_df = pd.DataFrame(overrides, columns=["column", "note"])
overrides_df.to_csv(os.path.join(ASSETS_TABLES, "eda01_ambiguous_overrides.csv"), index=False)
print(f"ambiguous-column overrides recorded: {len(overrides_df)}")
print(overrides_df.to_string(index=False))

ambiguous-column overrides recorded: 2
     column                                                                                               note
         id loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test
policy_code    single-valued category code in this population, not a measured quantity -- treat as categorical


**Result**

| column | note |
|---|---|
| `id` | loan identifier, not a numeric feature -- exclude from modeling despite passing the cast-rate test |
| `policy_code` | single-valued category code in this population, not a measured quantity -- treat as categorical |

Saved to `eda01_ambiguous_overrides.csv` — the cleaning notebook should check this file and treat these two by their note, not by cast-rate alone. This list is meant to grow: any later notebook that spots another miscategorized column should add it here rather than handling it locally.

**Next:** full null%/cardinality profile across every column.

## Cell 6 — Full null% / cardinality profile, every column

- DuckDB's `SUMMARIZE` gives a one-pass null%/min/max/approx-distinct profile per column, in a single query
- Run against `windowed`, covering the complete 151-column raw schema

**Answers:** which columns are the most/least populated, and are any fully null?

In [6]:
# one-pass null%/cardinality profile, every column, full list ordered by null% desc
summ = con.sql("SUMMARIZE windowed").df()
summ["null_pct"] = summ["null_percentage"].astype(float)
summ_sorted = summ[["column_name", "column_type", "null_pct", "approx_unique"]].sort_values("null_pct", ascending=False)
print(summ_sorted.to_string(index=False))
summ_sorted.to_csv(os.path.join(ASSETS_TABLES, "eda01_summ_sorted.csv"), index=False)
print()
print(f"columns with 0% nulls: {(summ_sorted['null_pct']==0).sum()} of {len(summ_sorted)}")
print(f"columns 100% null: {(summ_sorted['null_pct']==100).sum()}")

                               column_name column_type  null_pct  approx_unique
                                 member_id     VARCHAR    100.00              0
                              next_pymnt_d     VARCHAR    100.00              1
orig_projected_additional_accrued_interest     VARCHAR     99.69           3433
       sec_app_mths_since_last_major_derog     VARCHAR     99.65             93
                           hardship_length     VARCHAR     99.52              1
                           hardship_amount     VARCHAR     99.52           4136
                           hardship_status     VARCHAR     99.52              1
                             deferral_term     VARCHAR     99.52              1
                         hardship_end_date     VARCHAR     99.52             27
                           hardship_reason     VARCHAR     99.52              9
                       hardship_start_date     VARCHAR     99.52             27
                             hardship_ty

**Result**

| Metric | Value |
|---|---|
| Fully populated columns | 80 of 152 |
| Fully null columns | 2 (`member_id`, `next_pymnt_d`) |

- `member_id` is scrubbed by Lending Club before publication — dead weight, not signal
- The highest-missingness band (~99.5%+) is entirely hardship/settlement fields — worth understanding on their own terms, not dismissing as "mostly empty"

**Next:** whether that missingness is itself informative, or just structural noise.

## Cell 7 — Does high missingness carry signal?

- A field that's 99.5% null isn't automatically useless — if *whether* it's populated correlates with the outcome, the missingness itself is a feature
- Checking this directly rather than assuming it either way

**Answers:** do loans with a populated hardship/settlement record have a different bad rate than loans without one?

In [7]:
# does having a hardship record at all correlate with the outcome?
hardship_signal = con.sql("""
    SELECT (hardship_type IS NULL) AS hardship_type_is_null,
           count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM windowed GROUP BY 1 ORDER BY 1
""").df()
print("bad rate by whether hardship_type is populated:")
print(hardship_signal.to_string(index=False))
print()

# same check for the debt settlement flag
settlement_signal = con.sql("""
    SELECT debt_settlement_flag, count(*) AS n, round(avg(is_bad), 3) AS bad_rate
    FROM windowed GROUP BY 1 ORDER BY 1
""").df()
print("bad rate by debt_settlement_flag:")
print(settlement_signal.to_string(index=False))
print()

# hardship_flag itself -- check whether it actually varies in this population
hardship_flag_vals = con.sql("SELECT DISTINCT hardship_flag FROM windowed").df()
print(f"distinct hardship_flag values in windowed: {hardship_flag_vals['hardship_flag'].tolist()}")

hardship_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_hardship_signal.csv"), index=False)
settlement_signal.to_csv(os.path.join(ASSETS_TABLES, "eda01_settlement_signal.csv"), index=False)

bad rate by whether hardship_type is populated:
 hardship_type_is_null       n  bad_rate
                 False    5726     0.705
                  True 1190153     0.203

bad rate by debt_settlement_flag:
debt_settlement_flag       n  bad_rate
                   N 1163542     0.183
                   Y   32337     1.000

distinct hardship_flag values in windowed: ['N']


**Result**

| Group | Populated | n | Bad rate |
|---|---|---|---|
| Hardship record exists (`hardship_type` not null) | Yes | 5,726 | 70.5% |
| No hardship record | No | 1,190,153 | 20.3% |
| Debt settlement flag = Y | Yes | 32,337 | 100.0% |
| Debt settlement flag = N | No | 1,163,542 | 18.3% |

- `hardship_flag` is a dead end for this population — every row in `windowed` shows `N` (it marks a loan *currently* on an active hardship plan; a matured/finished loan is never "currently" anything)
- `hardship_type IS NULL` is the field that actually carries the history

**What these fields are**

| Field group | What it represents | Populated when |
|---|---|---|
| Hardship (`hardship_type`, `hardship_length`, `hardship_amount`, `hardship_start_date`/`hardship_end_date`, ...) | Temporary payment-relief program — reduced or paused payments for a borrower in financial distress | Loan entered a hardship plan |
| Settlement (`settlement_status`, `settlement_amount`, `settlement_percentage`, `settlement_term`, ...) | Negotiated payoff for less than the full balance, via a third-party settlement company | Loan is already severely delinquent |

Both are event-triggered — populated only for the subset of loans that entered that specific process, not the general population.

**Is the missingness itself a signal?**
- Yes, and a strong one
- Hardship record present → 3.5x the base bad rate (70.5% vs. 20.3%)
- Debt settlement flag = Y → 100.0% bad, by construction — settlement is a workout for loans already failing, not a trait observed before the outcome

**Should they be used in modeling?**
- Not as raw predictors — this is a leakage question, not a missingness question
- Both fields only exist because a loan started going bad; they document the outcome unfolding, not a borrower characteristic knowable at origination
- A model trained on `debt_settlement_flag` would just relearn "settled loans are bad" (already true by definition) — and a brand-new loan has no settlement history yet at scoring time anyway
- Same logic applies to `hardship_type` and its associated fields

**What they're good for instead, and when**

| Use | Stage | Notes |
|---|---|---|
| Binary "ever had a hardship/settlement event" indicator (from the null pattern) | Post-hoc, not origination-time | Collections prioritization, loss-given-default modeling on already-troubled loans |
| Exclude from PD feature set, with leakage reasoning documented | `03_data_cleaning/01_cleaning_and_feature_prep.ipynb` | Not just dropped for "too many nulls" |
| Revisit if target needs more granularity than binary `is_bad` | `02_eda/07_target_outcome_objective.ipynb` | Only if that need arises |

**How they'd be treated if ever used**
- The null pattern itself is the feature (`hardship_type IS NULL` as a 0/1 flag) — not the raw amount/date fields
- Those raw fields are only meaningful conditional on the event having happened; imputing them for the 99.5% of loans where nothing happened would fabricate values with no real meaning